In [24]:
import optuna.exceptions
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Load Data

In [26]:
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [27]:
dataset_path = '../dataset'

dataset = datasets.ImageFolder(root=dataset_path, transform=image_transforms)
len(dataset)

2300

In [28]:
dataset.classes

['F_Breakage', 'F_Crushed', 'F_Normal', 'R_Breakage', 'R_Crushed', 'R_Normal']

In [29]:
num_classes = len(dataset.classes)
num_classes

6

In [30]:
train_size = int(0.75*len(dataset))
val_size = len(dataset) - train_size

train_size, val_size

(1725, 575)

In [31]:
from torch.utils.data import random_split

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [32]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, pin_memory=True, num_workers=2)

## Model 5: EfficientNet V2-S with Hyper parameter tuning

In [33]:
class CarClassifierConvNeXt(nn.Module):
    def __init__(self, num_classes, dropout_rate):
        super().__init__()
        self.model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        # Freeze all layers except the final fully connected layer
        for param in self.model.parameters():
            param.requires_grad = False

        # Retrieve the in_features of the original classifier (which is a Sequential block)
        in_features = self.model.classifier[-1].in_features

        # Replace the final fully connected layer
        self.model.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),              # Pools spatial dimensions down to 1x1 -> [32, 768, 1, 1]
            nn.Flatten(start_dim=1, end_dim=-1),  # Flatten to 2D -> [32, 768]
            nn.LayerNorm(in_features, eps=1e-6),  # LayerNorm now safely receives [*, 768]
            nn.Linear(in_features, 256),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [34]:
def objective(trial):
    # Suggest values for the hyperparameters
    lr = trial.suggest_float('lr', 1e-4, 3e-3, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)

    # load the model
    model = CarClassifierConvNeXt(num_classes=num_classes, dropout_rate=dropout_rate).to(device)

    # define the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.model.classifier.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.2)
    scaler = torch.amp.GradScaler('cuda')

    # training the loop
    epochs = 15
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_num, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device)
            # Zero the parameter gradients
            optimizer.zero_grad()
            # Forward pass
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            # Backward pass
            scaler.scale(loss).backward()
            # Update the weights
            scaler.step(optimizer)
            scaler.update()

            if (batch_num+1) % 10 == 0:
                print(f"Batch: {batch_num+1}, Epoch: {epoch+1}, Loss: {loss.item():.2f}")

            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss/len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

    # Model Evaluation:

        model.eval()
        total = 0.0
        correct = 0.0
        val_loss = 0.0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        avg_val_loss = val_loss / len(val_loader.dataset)
        val_accuracy = 100 * correct / total

        trial.report(val_accuracy, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        # FIXED: Call the scheduler step at the end of the epoch using validation loss
        scheduler.step(avg_val_loss)

    return val_accuracy

In [36]:
import optuna

# 1. Add a pruner to kill bad hyperparameter trials early
# startup_trials=5: Allows the first 5 trials to run fully to establish a baseline performance median
# n_warmup_steps=3: Gives trials 3 epochs to warm up before they are eligible for pruning
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
)

# 2. Run the optimization
study.optimize(objective, n_trials=20)

# 3. Print the breakthrough results
print("Best trial visual baseline accuracy:")
print(f"  Value: {study.best_trial.value:.2f}%")
print("  Params: ")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

[I 2026-11-04 16:25:06,372] A new study created in memory with name: no-name-49943d0b-b387-40d5-a7eb-12d7d20d5b57
[W 2026-11-04 16:25:11,978] Trial 0 failed with parameters: {'lr': 0.0003574712922600243, 'dropout_rate': 0.5852142919229748} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "F:\PyCharm 2025.1.2\projects\.venv\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Aniruddh_G\AppData\Local\Temp\ipykernel_25396\2853636730.py", line 21, in objective
    for batch_num, (images, labels) in enumerate(train_loader):
                                       ~~~~~~~~~^^^^^^^^^^^^^^
  File "F:\PyCharm 2025.1.2\projects\.venv\.venv\Lib\site-packages\torch\utils\data\dataloader.py", line 505, in __iter__
    return self._get_iterator()
           ~~~~~~~~~~~~~~~~~~^^
  File "F:\PyCharm 2025.1.2\projects\.venv\.venv\Lib\site-packages\torch\utils\data\dataloader.py", line 

KeyboardInterrupt: 

In [ ]:
study.best_params